In [1]:
import sys
from pathlib import Path

root = Path().resolve()
sys.path.insert(0, str(root / "src"))


In [2]:
import tensorflow as tf

from FLRW_Net.network.network import NeuralNetwork

tf.keras.backend.set_floatx("float64")

In [27]:
flrw_net = NeuralNetwork(number_of_timesteps=7, triangulation="5-cell", cosmological_constant=1e-10)
adam_optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-6,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-7,
    amsgrad=False,
    decay=0.9,
    clipnorm=2,
    clipvalue=None,
    global_clipnorm=None,
)
flrw_net.compile(optimizer=adam_optimizer)

In [28]:
import numpy as np
inputs = tf.constant([[1, 2/5-3/8, 1.5, 2/5-3/8, 2, 2/5-3/8, 2.5, 2/5-3/8, 4, 2/5-3/8, 4.5, 2/5-3/8, 5, 2/5-3/8, 5.5]], dtype=tf.float64)  # 5-cell
# inputs = tf.constant([[1, 1/2-3/8, 1.5, 1/2-3/8, 2]], dtype=tf.float64)  # 16-cell
# inputs = tf.constant([[1, (3 + np.sqrt(5)) / 2, 2]], dtype=tf.float64)  # 600-cell
flrw_net.training(inputs, epochs=5000)

Training: 100%|█████████████████| 5000/5000 [04:10<00:00, 19.92step/s]


(1.489514989361066e-26,
 [2.760882125154657e-16,
  0.00035439808850348256,
  0.00012239228343267192,
  2.50536167648138e-05,
  9.752358772507415e-07,
  1.2056823806851712e-05,
  2.414619531087258e-05,
  2.8489169776746513e-05,
  2.6963915349717284e-05,
  2.221958312580694e-05,
  1.6498265118194973e-05,
  1.0864816366287909e-05,
  6.009471685133646e-06,
  2.4955585529115128e-06,
  5.665089516096846e-07,
  2.4464590643072375e-07,
  9.259950392198172e-07,
  1.6530023376958086e-06,
  2.049006457818063e-06,
  2.083939574362888e-06,
  1.8367348120222463e-06,
  1.4019266413451133e-06,
  8.939921577044853e-07,
  4.373370072527573e-07,
  1.439403873338126e-07,
  1.0202575500482469e-07,
  2.4577671046849196e-07,
  4.0147256462719e-07,
  4.818036592867321e-07,
  4.6255808906621297e-07,
  3.6083539738250994e-07,
  2.1804489185717658e-07,
  8.53976245929518e-08,
  1.2942999741905228e-08,
  2.6463132562945967e-08,
  8.081637214391235e-08,
  1.2302237745988443e-07,
  1.3125060458700038e-07,
  1.06320

In [29]:
for key, value in flrw_net.model_params._asdict().items():
    print(key, value.numpy())
tf.print(flrw_net(inputs))

n1 10.0
n2 10.0
n3 5.0
nte 3.0
lamb 1e-10
[[1 0.59292427477125886 1.9374955941394163 ... 5.4374987385003015 0.039529268595060306 5.5]]


In [6]:
from FLRW_Net.utils.losses import strut_losses, spatial_edge_losses

prediction = tf.constant([[1, 0.63246212021514625, 2]], dtype=tf.float64)
loss_struts = strut_losses(prediction, flrw_net.model_params)
loss_spatial_edges = spatial_edge_losses(prediction, flrw_net.model_params)
combined = tf.concat([loss_struts, loss_spatial_edges], axis=1)
loss = tf.squeeze(tf.reduce_mean(combined, axis=1))
tf.print(loss)

1.3244711362856916e-25


In [7]:
tf.print(tf.squeeze(tf.reduce_mean(tf.concat([strut_losses(prediction, flrw_net.model_params), spatial_edge_losses(prediction, flrw_net.model_params)], axis=1), axis=1)))

1.3244711362856916e-25


In [ ]:
# Check whether the forward feed gives the same result
import tensorflow as tf

from FLRW_Net.network.network import NeuralNetwork

from src.OneStep.layer import HiddenLayer as Layer1
from src.TwoStep.layer import HiddenLayer as Layer2
from src.ThreeStep.layer import HiddenLayer as Layer3
from src.FourStep.layer import HiddenLayer as Layer4

layer1 = Layer1()
layer2 = Layer2()
layer3 = Layer3()
layer4 = Layer4()
flrw_net_1 = NeuralNetwork(number_of_timesteps=1, triangulation="5-cell", cosmological_constant=1e-3)
flrw_net_2 = NeuralNetwork(number_of_timesteps=2, triangulation="5-cell", cosmological_constant=1e-3)
flrw_net_3 = NeuralNetwork(number_of_timesteps=3, triangulation="5-cell", cosmological_constant=1e-3)
flrw_net_4 = NeuralNetwork(number_of_timesteps=4, triangulation="5-cell", cosmological_constant=1e-3)

inputs_1 = tf.constant([[1, 0.67, 2]], dtype=tf.float64)
inputs_2 = tf.constant([[1, 0.67, 2, 0.68, 3]], dtype=tf.float64)
inputs_3 = tf.constant([[1, 0.67, 2, 0.68, 3, 0.69, 4]], dtype=tf.float64)
inputs_4 = tf.constant([[1, 0.67, 2, 0.68, 3, 0.69, 4, 0.7, 5]], dtype=tf.float64)

tf.print(layer1(inputs_1))
tf.print(flrw_net_1(inputs_1))
tf.print()
tf.print(layer2(inputs_2))
tf.print(flrw_net_2(inputs_2))
tf.print()
tf.print(layer3(inputs_3), summarize=-1)
tf.print(flrw_net_3(inputs_3))
tf.print()
tf.print(layer4(inputs_4), summarize=-1)
tf.print(flrw_net_4(inputs_4))